# Task 4: Sentiment Analysis with NLP

This notebook builds a sentiment classifier from synthetic review text generated from the supermarket dataset.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from wordcloud import WordCloud

data_path = 'c:/Users/Admin/OneDrive/Documents/BigData project/SuperMarket Analysis.csv'
pandas_df = pd.read_csv(data_path)

sentiment_df = pandas_df[['Rating', 'Branch', 'Product line', 'Payment', 'Gender']].copy()
sentiment_df['sentiment'] = (sentiment_df['Rating'] >= 7).astype(int)

def build_review(row):
    product = row['Product line']
    branch = row['Branch']
    payment = row['Payment']
    gender = row['Gender']
    if row['sentiment'] == 1:
        return f'Great {product.lower()} purchase at {branch} branch. {gender} used {payment.lower()} and enjoyed fast service and excellent value.'
    return f'Disappointing experience with {product.lower()} at {branch}. The {payment.lower()} checkout felt slow and the value did not meet expectations.'

sentiment_df['review_text'] = sentiment_df.apply(build_review, axis=1)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9]', ' ', text)
    tokens = [word for word in text.split() if word not in set(ENGLISH_STOP_WORDS) and len(word) > 2]
    return ' '.join(tokens)

sentiment_df['cleaned_text'] = sentiment_df['review_text'].apply(clean_text)
X_train, X_test, y_train, y_test = train_test_split(
    sentiment_df['cleaned_text'], sentiment_df['sentiment'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=200)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)
predictions = clf.predict(X_test_vec)

accuracy = accuracy_score(y_test, predictions)
print('Sentiment model accuracy:', round(accuracy, 3))
print(classification_report(y_test, predictions, target_names=['negative', 'positive']))
print('Confusion matrix:')
print(confusion_matrix(y_test, predictions))

positive_words = ' '.join(sentiment_df.loc[sentiment_df['sentiment'] == 1, 'cleaned_text'])
negative_words = ' '.join(sentiment_df.loc[sentiment_df['sentiment'] == 0, 'cleaned_text'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].set_title('Positive review words')
axes[0].imshow(WordCloud(width=400, height=200, background_color='white').generate(positive_words))
axes[0].axis('off')
axes[1].set_title('Negative review words')
axes[1].imshow(WordCloud(width=400, height=200, background_color='white').generate(negative_words))
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Notes

- Review text is synthetic because the source CSV does not contain raw customer feedback.
- The notebook uses a simple rating-based sentiment label and TF-IDF features.